## v1

PASO 1: Configuración del Entorno


En este paso preparamos todo el entorno de trabajo para un problema de segmentación de imágenes médicas, donde usaremos un modelo preentrenado (ResNet50) como extractor de características.

La idea es:

Definir parámetros consistentes (tamaño, batch, épocas, etc.)
Importar librerías correctas
Preparar el entorno para que ResNet entienda las imágenes correctamente
Crear una estructura ordenada de archivos

ResNet no recibe imágenes normales, necesita un preprocesamiento específico (preprocess_input), que se encargará de escalar y normalizar correctamente los datos.

In [ ]:
# -*- coding: utf-8 -*-
"""
PASO 1: Configuración del entorno y parámetros (Enfoque ResNet)

"""


import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob

import tensorflow as tf

# Keras
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D,
                                     UpSampling2D, Concatenate, BatchNormalization, Activation)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50

#  Preprocesamiento clave para ResNet
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Colab
from google.colab import files



# Tamaño estándar (requerido por ResNet)
IMG_SIZE = 224

# Batch size (balance entre velocidad y memoria)
BATCH_SIZE = 8

# Número de épocas
EPOCHS = 50

# Early stopping (evita sobreentrenamiento)
PATIENCE = 10

# Directorios
BASE_DIR = "/content/dataset/"
INPUT_DIR = os.path.join(BASE_DIR, "inputs/")
MASK_DIR = os.path.join(BASE_DIR, "masks/")
MODEL_DIR = "/content/model/"
RESULTS_DIR = "/content/results/"

#  CREAR ESTRUCTURA

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


#  VERIFICAR GPU

print(" Verificando GPU...")
print("GPU disponible:", tf.config.list_physical_devices('GPU'))


#  CONFIRMACIÓN


print("\n Paso 1 completado correctamente")
print(f" Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}")
print(f" Batch size: {BATCH_SIZE}")
print(f" Épocas: {EPOCHS}")
print(f" Dataset en: {BASE_DIR}")
print(f" Modelo se guardará en: {MODEL_DIR}")

paso 2: Carga de Imágenes y Construcción del Conjunto de Datos (Enfoque ResNet)


En esta etapa se construye el conjunto de datos que será utilizado para entrenar el modelo de segmentación.  aquí se establece una correspondencia uno a uno entre cada imagen de entrada y su respectiva máscara de salida.

El problema abordado corresponde a segmentación semántica supervisada, por lo que es indispensable contar con pares de datos correctamente alineados:

Imagen de entrada: fondo de ojo.
Imagen de salida (máscara): representación de la región de interés (copa/disco).

Dado que la disponibilidad de datos puede ser limitada, se incorporan técnicas de aumento de datos (data augmentation) que permiten generar variaciones geométricas (rotaciones y reflexiones) manteniendo la coherencia entre imagen y máscara. Esto mejora la capacidad de generalización del modelo sin introducir inconsistencias en las etiquetas.

Asimismo, se realiza un preprocesamiento estandarizado:

Redimensionamiento a un tamaño fijo compatible con ResNet (224×224).
Normalización de las imágenes de entrada mediante la función específica preprocess_input.
Escalamiento de las máscaras a valores en el rango [0,1].

Finalmente, los datos son organizados en tensores listos para el entrenamiento.

In [ ]:
# -*- coding: utf-8 -*-
"""
PASO 2: Carga de imágenes y construcción del dataset

Se cargan imágenes de entrada y sus máscaras correspondientes,
se preprocesan y se generan aumentos de datos consistentes.
"""

import os
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess


# SUBIDA DE IMÁGENES


print("Sube una imagen de entrada (fondo de ojo):")
uploaded_input = files.upload()
input_name = list(uploaded_input.keys())[0]

print("Sube la máscara correspondiente (copa/disco):")
uploaded_mask = files.upload()
mask_name = list(uploaded_mask.keys())[0]


#  PREPROCESAMIENTO

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = resnet_preprocess(img)  # normalización específica ResNet
    return img

def preprocess_mask(path):
    mask = cv2.imread(path, 0)  # escala de grises
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0
    mask = np.expand_dims(mask, axis=-1)
    return mask

# Cargar imágenes originales
input_img = preprocess_image(input_name)
mask_img = preprocess_mask(mask_name)

# DATA AUGMENTATION


def augment_data(image, mask):
    images = []
    masks = []

    # Rotaciones + flips (mucho más dataset)
    for angle in [0, 90, 180, 270]:
        img_rot = np.rot90(image, k=angle//90)
        mask_rot = np.rot90(mask, k=angle//90)

        # Original rotada
        images.append(img_rot)
        masks.append(mask_rot)

        # Flip horizontal
        images.append(np.fliplr(img_rot))
        masks.append(np.fliplr(mask_rot))

        # Flip vertical
        images.append(np.flipud(img_rot))
        masks.append(np.flipud(mask_rot))

    return images, masks
X, Y = augment_data(input_img, mask_img)

X = np.array(X)
Y = np.array(Y)

print("Dataset generado:")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


#  VISUALIZACIÓN


plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.title("Imagen de entrada")
plt.imshow((X[0] + 1)/2)  # reescalar para visualizar
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Máscara")
plt.imshow(Y[0].squeeze(), cmap='gray')
plt.axis("off")

plt.show()

Paso 3: Construcción del Modelo de Segmentación (U-Net con ResNet50 como Encoder)

En este paso se define la arquitectura del modelo de aprendizaje profundo que resolverá el problema de segmentación. Se adopta un enfoque híbrido basado en U-Net, integrando un modelo preentrenado (ResNet50) como encoder (extractor de características).

El encoder (ResNet50) permite aprovechar representaciones aprendidas previamente sobre grandes volúmenes de datos (ImageNet), lo que mejora la capacidad de generalización del modelo incluso con conjuntos de datos pequeños. Este componente reduce progresivamente la resolución espacial de la imagen mientras incrementa la abstracción semántica.

Por otro lado, el decoder se encarga de reconstruir la segmentación a resolución original mediante operaciones de upsampling. Se incorporan conexiones tipo skip connection entre encoder y decoder, lo que permite recuperar detalles espaciales finos perdidos durante el proceso de compresión.

Finalmente, el modelo produce una salida con activación sigmoide, adecuada para segmentación binaria (copa/disco vs fondo). Se utiliza la función de pérdida Binary Crossentropy, aunque en escenarios médicos es común complementar con métricas como Dice.

In [ ]:
# -*- coding: utf-8 -*-
"""
PASO 3: Modelo U-Net con ResNet50 + Loss combinada (BCE + Dice)
"""

import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, UpSampling2D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import tensorflow.keras.backend as K



def combined_loss(y_true, y_pred):
    # Binary Crossentropy
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    # Dice Loss
    smooth = 1e-6
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    dice = 1 - ((2. * intersection + smooth) /
                (K.sum(y_true_f) + K.sum(y_pred_f) + smooth))

    return bce + dice

#  MODELO

def build_model(img_size=224):

    inputs = Input(shape=(img_size, img_size, 3))

    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=inputs
    )



    for layer in base_model.layers[:-30]:
        layer.trainable = False

    for layer in base_model.layers[-30:]:
        layer.trainable = True

    c1 = base_model.get_layer("conv1_relu").output
    c2 = base_model.get_layer("conv2_block3_out").output
    c3 = base_model.get_layer("conv3_block4_out").output
    c4 = base_model.get_layer("conv4_block6_out").output
    c5 = base_model.get_layer("conv5_block3_out").output

    # DECODER


    u1 = UpSampling2D()(c5)
    u1 = Concatenate()([u1, c4])
    u1 = Conv2D(512, 3, padding="same", activation="relu")(u1)

    u2 = UpSampling2D()(u1)
    u2 = Concatenate()([u2, c3])
    u2 = Conv2D(256, 3, padding="same", activation="relu")(u2)

    u3 = UpSampling2D()(u2)
    u3 = Concatenate()([u3, c2])
    u3 = Conv2D(128, 3, padding="same", activation="relu")(u3)

    u4 = UpSampling2D()(u3)
    u4 = Concatenate()([u4, c1])
    u4 = Conv2D(64, 3, padding="same", activation="relu")(u4)

    u5 = UpSampling2D()(u4)
    u5 = Conv2D(32, 3, padding="same", activation="relu")(u5)

    outputs = Conv2D(1, 1, activation="sigmoid")(u5)

    model = Model(inputs, outputs)

    return model

#  COMPILACIÓN


model = build_model(IMG_SIZE)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=['accuracy']
)


#  RESUMEN


model.summary()

Paso 4: Entrenamiento del Modelo y Estrategias de Regularización

En esta etapa se lleva a cabo el proceso de entrenamiento del modelo previamente definido. El objetivo es optimizar los parámetros entrenables del decoder para que el modelo aprenda a mapear correctamente las imágenes de entrada hacia sus respectivas máscaras de segmentación.

Dado que se dispone de un conjunto de datos reducido, es fundamental implementar estrategias que eviten el sobreajuste y mejoren la estabilidad del entrenamiento. Para ello se emplean técnicas como:

Early Stopping: detiene el entrenamiento automáticamente cuando la función de pérdida en validación deja de mejorar durante un número determinado de épocas.
Model Checkpoint: guarda el modelo con mejor desempeño durante el entrenamiento.
División del dataset: separación en subconjuntos de entrenamiento y validación para evaluar la capacidad de generalización.

Adicionalmente, se monitorean métricas relevantes durante el entrenamiento, lo que permite analizar el comportamiento del modelo y detectar posibles problemas como subentrenamiento o sobreentrenamiento.

In [ ]:
# -*- coding: utf-8 -*-
"""
PASO 4: Entrenamiento del modelo (CORREGIDO)
"""

from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt


#  PARÁMETROS DE ENTRENAMIENTO


EPOCHS = 100
BATCH_SIZE = 8


#  DIVISIÓN DEL DATASET

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.25, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_DIR + "best_model.h5",
    monitor='val_loss',
    save_best_only=True
)


# ENTRENAMIENTO

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, checkpoint]
)


#  VISUALIZACIÓN

plt.figure()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss")

plt.show()

paso 5

En esta fase final se evalúa el desempeño del modelo entrenado mediante la generación de predicciones sobre datos de entrada. El objetivo es verificar si el modelo ha aprendido correctamente la correspondencia entre la imagen de fondo de ojo y la máscara de segmentación (copa/disco).

A diferencia de métricas numéricas únicamente, en problemas de segmentación médica es fundamental realizar una evaluación cualitativa, es decir, comparar visualmente:

Imagen de entrada
Máscara real (ground truth)
Máscara predicha por el modelo

Esto permite identificar si el modelo segmenta correctamente las regiones de interés, si presenta errores de forma, tamaño o localización, o si simplemente no ha logrado aprender el patrón.

Adicionalmente, se aplica un umbral sobre la salida del modelo (activación sigmoide) para convertir probabilidades en una máscara binaria interpretable.

In [ ]:
# -*- coding: utf-8 -*-
"""
PASO 5: Evaluación y predicción (CORREGIDO FINAL)
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K


# FUNCIÓN LOSS (OBLIGATORIA PARA CARGAR)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    smooth = 1e-6
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    dice = 1 - ((2. * intersection + smooth) /
                (K.sum(y_true_f) + K.sum(y_pred_f) + smooth))

    return bce + dice


model = tf.keras.models.load_model(
    MODEL_DIR + "best_model.h5",
    custom_objects={'combined_loss': combined_loss}
)


idx = 0
input_sample = X_val[idx]
true_mask = Y_val[idx]

pred = model.predict(np.expand_dims(input_sample, axis=0))[0]

#  Umbral más flexible
pred_mask = (pred > 0.3).astype(np.float32)




def deprocess_image(img):
    img = img.copy()

    img[..., 0] += 103.939
    img[..., 1] += 116.779
    img[..., 2] += 123.68

    img = img[..., ::-1]
    img = np.clip(img / 255.0, 0, 1)

    return img


# VISUALIZACIÓN


plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Imagen de entrada")
plt.imshow(deprocess_image(input_sample))
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Máscara real")
plt.imshow(true_mask.squeeze(), cmap='gray')
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Máscara predicha")
plt.imshow(pred_mask.squeeze(), cmap='gray')
plt.axis("off")

plt.show()

## v2

In [ ]:
!unzip exp_1_comp.zip

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
import random
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate, BatchNormalization, Activation, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import add
from tensorflow.keras.layers import multiply
from tensorflow.keras.callbacks import ReduceLROnPlateau

# ==========================================
# PASO 1: CONFIGURACIÓN
# ==========================================

IMG_SIZE = 512  # Actualizado a 512x512
BATCH_SIZE = 8  # Reducido para mitigar el consumo de memoria RAM/VRAM
EPOCHS = 50
NUM_CLASSES = 3 # 0: Fondo, 1: Disco, 2: Copa

BASE_DIR = "/content/exp_1_comp/cropped/"
IMAGE_DIR = os.path.join(BASE_DIR, "image_cropped/")
MASK_DIR = os.path.join(BASE_DIR, "mask_cropped/")
MODEL_DIR = "/content/model/"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# ==========================================
# PASO 2: FUNCIONES DE PREPROCESAMIENTO Y MODELO
# ==========================================

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

def preprocess_multiclass_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask_classes = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    mask_classes[mask < 50] = 2       # Negro -> Copa Óptica
    mask_classes[(mask >= 50) & (mask <= 200)] = 1  # Gris -> Disco Óptico
    mask_classes[mask > 200] = 0      # Blanco -> Fondo

    mask_one_hot = tf.keras.utils.to_categorical(mask_classes, num_classes=NUM_CLASSES)
    return mask_one_hot

def weighted_multiclass_combined_loss(y_true, y_pred):
    # Pesos de clase: Fondo=0.1, Disco=1.0, Copa=2.0 (La copa es la más difícil/pequeña)
    class_weights = tf.constant([0.5, 1.0, 1.5])

    # Entropía cruzada ponderada
    cce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    weight_map = tf.reduce_sum(class_weights * y_true, axis=-1)
    weighted_cce = K.mean(cce * weight_map)

    # Dice (como lo tenías, ya maneja el desequilibrio per se)
    smooth = 1e-6
    intersection = K.sum(y_true * y_pred, axis=[1, 2])
    union = K.sum(y_true, axis=[1, 2]) + K.sum(y_pred, axis=[1, 2])
    dice_per_class = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
    dice = K.mean(dice_per_class)

    # Combinación: Damos más importancia al CCE ponderado para delinear mejor los bordes
    return (0.7 * weighted_cce) + (0.3 * dice)

def attention_block(x, g, inter_shape):
    theta_x = Conv2D(inter_shape, 1, padding='same')(x)

    # Transformar g (señal de gating del decoder)
    phi_g = Conv2D(inter_shape, 1, padding='same')(g)

    # Sumar y aplicar no linealidad para obtener el mapa de atención
    concat_xg = add([phi_g, theta_x])
    act_xg = Activation('relu')(concat_xg)

    # Calcular coeficientes de atención (pesos)
    psi = Conv2D(1, 1, padding='same')(act_xg)
    sigmoid_xg = Activation('sigmoid')(psi)

    # Multiplicar las características originales x por los pesos de atención
    # Como ya son del mismo tamaño, no necesitamos el UpSampling2D aquí
    y = multiply([sigmoid_xg, x])

    # Devolver las características filtradas
    result = Conv2D(K.int_shape(x)[3], 1, padding='same')(y)
    result = BatchNormalization()(result)

    return result

def res_block(input_tensor, num_filters):
    # Primera convolución
    x = Conv2D(num_filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    # Segunda convolución
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)

    # Conexión "Atajo" (Skip connection) para la suma residual
    # Usamos una convolución 1x1 por si el número de filtros cambió
    shortcut = Conv2D(num_filters, 1, padding="same")(input_tensor)
    shortcut = BatchNormalization()(shortcut)

    # Sumamos el atajo a la salida de las convoluciones
    x = add([x, shortcut])
    x = Activation("relu")(x)
    return x

def build_attention_res_unet(img_size=512, num_classes=3):
    inputs = Input(shape=(img_size, img_size, 3))

    # ENCODER (Igual que antes)
    c1 = res_block(inputs, 32)
    p1 = MaxPooling2D()(c1)

    c2 = res_block(p1, 64)
    p2 = MaxPooling2D()(c2)

    c3 = res_block(p2, 128)
    p3 = MaxPooling2D()(c3)

    c4 = res_block(p3, 256)
    p4 = MaxPooling2D()(c4)
    p4 = Dropout(0.3)(p4)

    # BOTTLENECK
    b = res_block(p4, 512)
    b = Dropout(0.3)(b)

    # DECODER CON ATENCIÓN
    u1 = UpSampling2D()(b)
    att1 = attention_block(c4, u1, 128) # x=c4, g=u1
    concat1 = Concatenate()([u1, att1])
    c5 = res_block(concat1, 256)

    u2 = UpSampling2D()(c5)
    att2 = attention_block(c3, u2, 64)
    concat2 = Concatenate()([u2, att2])
    c6 = res_block(concat2, 128)

    u3 = UpSampling2D()(c6)
    att3 = attention_block(c2, u3, 32)
    concat3 = Concatenate()([u3, att3])
    c7 = res_block(concat3, 64)

    u4 = UpSampling2D()(c7)
    att4 = attention_block(c1, u4, 16)
    concat4 = Concatenate()([u4, att4])
    c8 = res_block(concat4, 32)

    outputs = Conv2D(num_classes, 1, activation="softmax")(c8)
    return Model(inputs, outputs)

In [ ]:
# ==========================================
# PASO 3: DIVISIÓN DE RUTAS (No de imágenes)
# ==========================================

# 1. Obtener todas las rutas completas
all_image_paths = sorted([os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
all_mask_paths = sorted([os.path.join(MASK_DIR, f) for f in os.listdir(MASK_DIR) if f.endswith(('.png', '.jpg', '.jpeg','.bmp'))])

# 2. Dividir las RUTAS en Train, Val, Test
X_temp_paths, X_test_paths, Y_temp_paths, Y_test_paths = train_test_split(
    all_image_paths, all_mask_paths, test_size=0.15, random_state=42
)

X_train_paths, X_val_paths, Y_train_paths, Y_val_paths = train_test_split(
    X_temp_paths, Y_temp_paths, test_size=0.20, random_state=42
)

print(f"Rutas de Entrenamiento: {len(X_train_paths)}")
print(f"Rutas de Validación: {len(X_val_paths)}")
print(f"Rutas de Pruebas (Test): {len(X_test_paths)}")

In [ ]:
# ==========================================
# PASO 4: CREACIÓN DEL GENERADOR (SEQUENCE)
# ==========================================

class RetinaGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_paths, mask_paths, batch_size, img_size, num_classes, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.img_size = img_size
        self.num_classes = num_classes
        self.augment = augment
        self.indexes = np.arange(len(self.image_paths))
        self.on_epoch_end() # Barajar datos al inicio

    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        batch_x_paths = [self.image_paths[k] for k in indexes]
        batch_y_paths = [self.mask_paths[k] for k in indexes]

        X, Y = self.__data_generation(batch_x_paths, batch_y_paths)
        return X, Y

    def __data_generation(self, batch_x_paths, batch_y_paths):
        X = np.empty((self.batch_size, self.img_size, self.img_size, 3), dtype=np.float32)
        Y = np.empty((self.batch_size, self.img_size, self.img_size, self.num_classes), dtype=np.float32)

        for i, (img_path, mask_path) in enumerate(zip(batch_x_paths, batch_y_paths)):
            img = preprocess_image(img_path)
            mask = preprocess_multiclass_mask(mask_path)

            if self.augment:
                k = random.choice([0, 1, 2, 3])
                if k > 0:
                    img = np.rot90(img, k=k)
                    mask = np.rot90(mask, k=k)
                if random.random() > 0.5:
                    img = np.fliplr(img)
                    mask = np.fliplr(mask)
                if random.random() > 0.5:
                    img = np.flipud(img)
                    mask = np.flipud(mask)

                if random.random() > 0.5:
                    # Ajuste de brillo aleatorio entre -30 y +30
                    factor_brillo = random.uniform(-0.1, 0.1)
                    img = tf.clip_by_value(img + factor_brillo, 0.0, 1.0).numpy()

                if random.random() > 0.5:
                    # Ajuste de contraste (multiplicador)
                    factor_contraste = random.uniform(0.8, 1.2)
                    # Centramos en 0.5 para no desfasar los colores, escalamos y devolvemos a 0.5
                    img = tf.clip_by_value((img - 0.5) * factor_contraste + 0.5, 0.0, 1.0).numpy()

            X[i,] = img
            Y[i,] = mask

        return X, Y

# Instanciamos los generadores
train_gen = RetinaGenerator(X_train_paths, Y_train_paths, BATCH_SIZE, IMG_SIZE, NUM_CLASSES, augment=True)
val_gen = RetinaGenerator(X_val_paths, Y_val_paths, BATCH_SIZE, IMG_SIZE, NUM_CLASSES, augment=False)

In [ ]:
# ==========================================
# PASO 5: ENTRENAMIENTO CON EL GENERADOR
# ==========================================
model = build_attention_res_unet(IMG_SIZE, NUM_CLASSES)

# Mantenemos el LR inicial de 1e-4
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss=weighted_multiclass_combined_loss, # Usa la pérdida que te funcionaba bien
              metrics=['accuracy'])

# Le damos 20 epochs de paciencia. Si se estanca un poco, que siga intentando.
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
checkpoint = ModelCheckpoint(filepath=MODEL_DIR + "best_unet_multiclass.h5", monitor='val_loss', save_best_only=True)

print("\nIniciando Entrenamiento...")

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS, # Asegúrate de que esto sea al menos 50 o 100
    callbacks=[early_stop, checkpoint] # Quitamos el reduce_lr
)

In [ ]:
# ==========================================
# PASO 6: EVALUACIÓN VISUAL Y MÉTRICAS (CON ELIPSE)
# ==========================================

def suavizado_organico(pred_mask, img_size):
    # Lienzo final en negro (Fondo = 0)
    final_mask = np.zeros((img_size, img_size), dtype=np.uint8)

    # IMPORTANTE: Creamos un "pincel" circular/elíptico, NO cuadrado.
    # El tamaño (35, 35) es la "fuerza". A mayor número, más liso queda.
    kernel_size = 35
    kernel_redondo = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    # ==========================================
    # 1. PROCESAR EL DISCO ÓPTICO (Clase 1 + 2)
    # ==========================================
    disco_binario = np.uint8((pred_mask == 1) | (pred_mask == 2)) * 255
    contornos_disco, _ = cv2.findContours(disco_binario, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contornos_disco:
        # Extraer solo la masa principal (elimina el ruido del lente izquierdo)
        contorno_mayor_disco = max(contornos_disco, key=cv2.contourArea)
        mascara_limpia_disco = np.zeros_like(disco_binario)
        cv2.drawContours(mascara_limpia_disco, [contorno_mayor_disco], -1, 255, -1)

        # Clausura: Rellena grietas o agujeros hacia adentro
        mascara_limpia_disco = cv2.morphologyEx(mascara_limpia_disco, cv2.MORPH_CLOSE, kernel_redondo)
        # Apertura: Lija las puntas y picos hacia afuera
        mascara_limpia_disco = cv2.morphologyEx(mascara_limpia_disco, cv2.MORPH_OPEN, kernel_redondo)

        # Plasmar en el lienzo final (Clase 1)
        final_mask[mascara_limpia_disco == 255] = 1

    # ==========================================
    # 2. PROCESAR LA COPA ÓPTICA (Clase 2)
    # ==========================================
    copa_binaria = np.uint8(pred_mask == 2) * 255
    contornos_copa, _ = cv2.findContours(copa_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contornos_copa:
        contorno_mayor_copa = max(contornos_copa, key=cv2.contourArea)
        mascara_limpia_copa = np.zeros_like(copa_binaria)
        cv2.drawContours(mascara_limpia_copa, [contorno_mayor_copa], -1, 255, -1)

        # Mismo tratamiento morfológico para la copa
        mascara_limpia_copa = cv2.morphologyEx(mascara_limpia_copa, cv2.MORPH_CLOSE, kernel_redondo)
        mascara_limpia_copa = cv2.morphologyEx(mascara_limpia_copa, cv2.MORPH_OPEN, kernel_redondo)

        # Plasmar en el lienzo final (Clase 2, sobreescribiendo el centro del disco)
        final_mask[mascara_limpia_copa == 255] = 2

    return final_mask

def visualize_prediction(model, input_image, true_mask_one_hot):
    pred = model.predict(np.expand_dims(input_image, axis=0), verbose=0)[0]
    pred_class = np.argmax(pred, axis=-1).astype(np.uint8)
    true_class = np.argmax(true_mask_one_hot, axis=-1)

    # 1. Filtro Mediano
    pred_smooth = cv2.medianBlur(pred_class, 29)

    # 2. Operación de Clausura
    kernel = np.ones((11,11), np.uint8)
    pred_smooth = cv2.morphologyEx(pred_smooth, cv2.MORPH_CLOSE, kernel)

    # 3. --- NUEVO: Ajuste de Elipse ---
    # Le pasamos la imagen suavizada para que los contornos sean más limpios
    pred_ellipse = suavizado_organico(pred_smooth, input_image.shape[0])

    plt.figure(figsize=(25, 5)) # Más ancha para 5 imágenes

    plt.subplot(1, 5, 1)
    plt.title("Imagen de Entrada")
    plt.imshow(input_image)
    plt.axis("off")

    plt.subplot(1, 5, 2)
    plt.title("Máscara Real")
    plt.imshow(true_class, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.subplot(1, 5, 3)
    plt.title("Predicción (Cruda)")
    plt.imshow(pred_class, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.subplot(1, 5, 4)
    plt.title("Predicción (Suavizada)")
    plt.imshow(pred_smooth, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.subplot(1, 5, 5)
    plt.title("Predicción (Elipse Final)")
    plt.imshow(pred_ellipse, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.show()

def calculate_iou(y_true, y_pred, num_classes):
    iou_list = []
    for c in range(1, num_classes):
        true_class = (y_true == c)
        pred_class = (y_pred == c)

        intersection = np.logical_and(true_class, pred_class).sum()
        union = np.logical_or(true_class, pred_class).sum()

        if union == 0:
            iou = 1.0 if intersection == 0 else 0.0
        else:
            iou = intersection / union
        iou_list.append(iou)
    return iou_list

print("\nEvaluando en el Set de Pruebas (Test)...")
iou_disco_crudo, iou_copa_cruda = [], []
iou_disco_final, iou_copa_final = [], []

# Iteramos manualmente sobre las rutas de Test para evaluar una por una
for img_path, mask_path in zip(X_test_paths, Y_test_paths):
    img = preprocess_image(img_path)
    mask = preprocess_multiclass_mask(mask_path)

    # Predecir
    pred = model.predict(np.expand_dims(img, axis=0), verbose=0)[0]

    # Extraer clases (Cruda)
    pred_class = np.argmax(pred, axis=-1).astype(np.uint8)
    true_class = np.argmax(mask, axis=-1)

    # --- Aplicar Post-procesamiento para Métricas Finales ---
    pred_smooth_eval = cv2.medianBlur(pred_class, 29)
    kernel_eval = np.ones((11,11), np.uint8)
    pred_smooth_eval = cv2.morphologyEx(pred_smooth_eval, cv2.MORPH_CLOSE, kernel_eval)
    pred_ellipse_eval = suavizado_organico(pred_smooth_eval, img.shape[0])
    # --------------------------------------------------------

    # Calcular métricas Crudas (como lo tenías antes)
    ious_crudos = calculate_iou(true_class, pred_class, NUM_CLASSES)
    iou_disco_crudo.append(ious_crudos[0])
    iou_copa_cruda.append(ious_crudos[1])

    # Calcular métricas Finales (Con Elipse)
    ious_finales = calculate_iou(true_class, pred_ellipse_eval, NUM_CLASSES)
    iou_disco_final.append(ious_finales[0])
    iou_copa_final.append(ious_finales[1])

print("--- RESULTADOS IOU ---")
print(f"Crudo - Disco Óptico: {np.mean(iou_disco_crudo):.4f} | Copa Óptica: {np.mean(iou_copa_cruda):.4f}")
print(f"FINAL - Disco Óptico: {np.mean(iou_disco_final):.4f} | Copa Óptica: {np.mean(iou_copa_final):.4f}")

# Mostrar predicción de un elemento aleatorio del set de Test
print("\nVisualización de ejemplo del Test Set:")
idx_random = random.randint(0, len(X_test_paths) - 1)
test_img = preprocess_image(X_test_paths[idx_random])
test_mask = preprocess_multiclass_mask(Y_test_paths[idx_random])

visualize_prediction(model, test_img, test_mask)

# v3 (transfer learning)

In [ ]:
!apt-get install -y p7zip-full

!7z x cropped.7z -o./content

In [ ]:
!pip install segmentation-models

In [ ]:
!pip install albumentations
import albumentations as A

In [ ]:
import os

os.environ["SM_FRAMEWORK"] = "tf.keras"

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import segmentation_models as sm

# ==========================================
# PASO 1: CONFIGURACIÓN
# ==========================================

IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 50
NUM_CLASSES = 3 # 0: Fondo, 1: Disco, 2: Copa

# Seleccionamos nuestro "Cerebro" pre-entrenado
BACKBONE = 'resnet18'
preprocess_input = sm.get_preprocessing(BACKBONE)

BASE_DIR = "/content/content/cropped/"
IMAGE_DIR = os.path.join(BASE_DIR, "image_cropped/")
MASK_DIR = os.path.join(BASE_DIR, "mask_cropped/")
MODEL_DIR = "/content/model/"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# ==========================================
# PASO 2: FUNCIONES BASE Y LOSS
# ==========================================

def preprocess_image_raw(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.uint8)

    # CLAHE: Ilumina los detalles ocultos sin dañar el color
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    img_enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return img_enhanced.astype(np.float32)

def preprocess_multiclass_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask_classes = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    mask_classes[mask < 50] = 2       # Negro -> Copa Óptica
    mask_classes[(mask >= 50) & (mask <= 200)] = 1  # Gris -> Disco Óptico
    mask_classes[mask > 200] = 0      # Blanco -> Fondo

    mask_one_hot = tf.keras.utils.to_categorical(mask_classes, num_classes=NUM_CLASSES)
    return mask_one_hot

def weighted_multiclass_combined_loss(y_true, y_pred):
    # Pesos de clase: Fondo=0.5, Disco=1.0, Copa=1.5
    class_weights = tf.constant([0.5, 1.0, 1.5])

    cce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    weight_map = tf.reduce_sum(class_weights * y_true, axis=-1)
    weighted_cce = K.mean(cce * weight_map)

    smooth = 1e-6
    intersection = K.sum(y_true * y_pred, axis=[1, 2])
    union = K.sum(y_true, axis=[1, 2]) + K.sum(y_pred, axis=[1, 2])
    dice_per_class = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
    dice = K.mean(dice_per_class)

    return (0.7 * weighted_cce) + (0.3 * dice)

In [ ]:
# ==========================================
# PASO 3: DIVISIÓN DE RUTAS
# ==========================================

all_image_paths = sorted([os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
all_mask_paths = sorted([os.path.join(MASK_DIR, f) for f in os.listdir(MASK_DIR) if f.endswith(('.png', '.jpg', '.jpeg','.bmp'))])

X_temp_paths, X_test_paths, Y_temp_paths, Y_test_paths = train_test_split(
    all_image_paths, all_mask_paths, test_size=0.15, random_state=42
)

X_train_paths, X_val_paths, Y_train_paths, Y_val_paths = train_test_split(
    X_temp_paths, Y_temp_paths, test_size=0.20, random_state=42
)

print(f"Rutas de Entrenamiento: {len(X_train_paths)}")
print(f"Rutas de Validación: {len(X_val_paths)}")
print(f"Rutas de Pruebas (Test): {len(X_test_paths)}")

In [ ]:
# ==========================================
# PASO 4: GENERADOR CON ALBUMENTATIONS
# ==========================================

# Definimos el "Pipeline" de mutaciones
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    # Deformación Espacial: Estira y encoje el ojo imitando variaciones anatómicas
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=45, p=0.5),
    # Deformación Biológica 1: Crea un efecto de cuadrícula ondulada
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    # Deformación Biológica 2: Estira localmente la imagen (El secreto de la U-Net original)
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    # Ajustes de luz y contraste simulando diferentes cámaras de fondo de ojo
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5)
])

class RetinaGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_paths, mask_paths, batch_size, img_size, num_classes, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.img_size = img_size
        self.num_classes = num_classes
        self.augment = augment
        self.indexes = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        batch_x_paths = [self.image_paths[k] for k in indexes]
        batch_y_paths = [self.mask_paths[k] for k in indexes]

        X, Y = self.__data_generation(batch_x_paths, batch_y_paths)
        return X, Y

    def __data_generation(self, batch_x_paths, batch_y_paths):
        X = np.empty((self.batch_size, self.img_size, self.img_size, 3), dtype=np.float32)
        Y = np.empty((self.batch_size, self.img_size, self.img_size, self.num_classes), dtype=np.float32)

        for i, (img_path, mask_path) in enumerate(zip(batch_x_paths, batch_y_paths)):
            img = preprocess_image_raw(img_path)
            mask = preprocess_multiclass_mask(mask_path)

            # --- APLICACIÓN DE ALBUMENTATIONS ---
            if self.augment:
                # Albumentations requiere que las clases de la máscara no estén en One-Hot aún
                mask_classes = np.argmax(mask, axis=-1).astype(np.uint8)

                # Aplicamos la transformación a la imagen y a la máscara simultáneamente
                augmented = train_transform(image=img, mask=mask_classes)
                img = augmented['image']
                mask_aug = augmented['mask']

                # Volvemos a convertir la máscara mutada a One-Hot
                mask = tf.keras.utils.to_categorical(mask_aug, num_classes=self.num_classes)

            # El pre-procesamiento del Backbone (ResNet, etc) siempre va AL FINAL
            img = preprocess_input(img)

            X[i,] = img
            Y[i,] = mask

        return X, Y

# Instanciamos los generadores (asegurando que augment=True está en train)
train_gen = RetinaGenerator(X_train_paths, Y_train_paths, BATCH_SIZE, IMG_SIZE, NUM_CLASSES, augment=True)
val_gen = RetinaGenerator(X_val_paths, Y_val_paths, BATCH_SIZE, IMG_SIZE, NUM_CLASSES, augment=False)

In [ ]:
# ==========================================
# PASO 5: ENTRENAMIENTO CON TRANSFER LEARNING (FPN)
# ==========================================

# 1. ¡AQUÍ ESTÁ LA NUEVA LÍNEA! Creamos la arquitectura FPN con ResNet18
model = sm.FPN(BACKBONE, classes=NUM_CLASSES, activation='softmax', encoder_weights='imagenet')

# 2. Definimos las funciones de pérdida expertas (Focal + Dice)
dice_loss = sm.losses.DiceLoss(class_weights=np.array([0.5, 1.0, 1.5]))
focal_loss = sm.losses.CategoricalFocalLoss()
total_loss = dice_loss + (1 * focal_loss)

# 3. Compilamos el modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss=total_loss,
              metrics=['accuracy', sm.metrics.IOUScore(threshold=0.5)])

# 4. Configuramos las paradas de emergencia y el guardado automático
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
# OJO: Cambié el nombre del archivo para que guardes el modelo FPN por separado
checkpoint = ModelCheckpoint(filepath=MODEL_DIR + "best_fpn_resnet18.h5", monitor='val_loss', save_best_only=True)

print(f"\nIniciando Entrenamiento FPN con Transfer Learning ({BACKBONE})...")

# 5. ¡A entrenar!
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    # TRUCO: Normalmente serían 34 pasos (272/8). Le pedimos 100 pasos.
    # El generador simplemente volverá a empezar, inyectando nuevas mutaciones continuamente.
    steps_per_epoch=100,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
# ==========================================
# PASO 6: EVALUACIÓN VISUAL, IOU Y MÉTRICA CLÍNICA (CDR)
# ==========================================

def limpieza_minimalista(pred_mask, img_size):
    # Versión que extrae solo la isla principal para limpiar ruido
    final_mask = np.zeros((img_size, img_size), dtype=np.uint8)

    # 1. DISCO ÓPTICO
    disco_binario = np.uint8((pred_mask == 1) | (pred_mask == 2)) * 255
    contornos_disco, _ = cv2.findContours(disco_binario, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos_disco:
        contorno_mayor_disco = max(contornos_disco, key=cv2.contourArea)
        cv2.drawContours(final_mask, [contorno_mayor_disco], -1, 1, -1)

    # 2. COPA ÓPTICA
    copa_binaria = np.uint8(pred_mask == 2) * 255
    contornos_copa, _ = cv2.findContours(copa_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos_copa:
        contorno_mayor_copa = max(contornos_copa, key=cv2.contourArea)
        cv2.drawContours(final_mask, [contorno_mayor_copa], -1, 2, -1)

    return final_mask

def calcular_cdr_mejorado(mask_2d):
    # Calcula la relación Copa-Disco (Vertical) agrupando fragmentos
    disco_bin = np.uint8((mask_2d == 1) | (mask_2d == 2))
    copa_bin = np.uint8(mask_2d == 2)

    contornos_disco, _ = cv2.findContours(disco_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contornos_copa, _ = cv2.findContours(copa_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h_disco, h_copa = 0, 0

    if contornos_disco:
        # Agrupamos TODOS los puntos de todos los fragmentos del disco
        todos_puntos_disco = np.vstack(contornos_disco)
        _, _, _, h_disco = cv2.boundingRect(todos_puntos_disco)

    if contornos_copa:
        # LA MAGIA: Agrupamos TODOS los fragmentos de la copa (ignorando el hueco de la vena)
        todos_puntos_copa = np.vstack(contornos_copa)
        _, _, _, h_copa = cv2.boundingRect(todos_puntos_copa)

    if h_disco > 0:
        return h_copa / h_disco
    return 0.0

def visualize_prediction(model, raw_image, true_mask_one_hot):
    input_model = preprocess_input(np.copy(raw_image))
    pred = model.predict(np.expand_dims(input_model, axis=0), verbose=0)[0]

    # Mismos umbrales inteligentes que usamos en la evaluación
    prob_disco = pred[:, :, 1]
    prob_copa = pred[:, :, 2]
    pred_class = np.zeros((raw_image.shape[0], raw_image.shape[1]), dtype=np.uint8)
    pred_class[prob_disco > 0.40] = 1
    pred_class[prob_copa > 0.33] = 2

    true_class = np.argmax(true_mask_one_hot, axis=-1)

    # Solo usamos la limpieza minimalista (sin blur ni morphologyEx)
    pred_final = limpieza_minimalista(pred_class, raw_image.shape[0])

    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.title("Imagen de Entrada")
    plt.imshow(raw_image / 255.0)
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.title("Máscara Real")
    plt.imshow(true_class, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.title("Predicción (Cruda)")
    plt.imshow(pred_class, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.title("Predicción (Limpia Minimalista)")
    plt.imshow(pred_final, cmap='gray', vmin=0, vmax=2)
    plt.axis("off")

    plt.show()

def calculate_iou(y_true, y_pred, num_classes):
    iou_list = []
    for c in range(1, num_classes):
        true_class = (y_true == c)
        pred_class = (y_pred == c)
        intersection = np.logical_and(true_class, pred_class).sum()
        union = np.logical_or(true_class, pred_class).sum()
        if union == 0:
            iou = 1.0 if intersection == 0 else 0.0
        else:
            iou = intersection / union
        iou_list.append(iou)
    return iou_list

print("\nEvaluando con TTA y Análisis Clínico (CDR)...")
iou_disco_crudo, iou_copa_cruda = [], []
iou_disco_final, iou_copa_final = [], []

cdr_reales = []
cdr_predichos = []
errores_cdr = []

for img_path, mask_path in zip(X_test_paths, Y_test_paths):
    raw_img = preprocess_image_raw(img_path)
    mask = preprocess_multiclass_mask(mask_path)
    true_class = np.argmax(mask, axis=-1)

    # --- INICIO DE TTA ---
    preds_tta = []
    for k in range(4):
        img_rot = np.rot90(raw_img, k=k)
        input_model = preprocess_input(np.copy(img_rot))
        pred_rot = model.predict(np.expand_dims(input_model, axis=0), verbose=0)[0]
        pred_inv = np.rot90(pred_rot, k=-k)
        preds_tta.append(pred_inv)

    pred_avg = np.mean(preds_tta, axis=0)
    prob_disco = pred_avg[:, :, 1]
    prob_copa = pred_avg[:, :, 2]

    pred_class = np.zeros((raw_img.shape[0], raw_img.shape[1]), dtype=np.uint8)
    pred_class[prob_disco > 0.40] = 1
    pred_class[prob_copa > 0.33] = 2
    # --- FIN DE TTA ---

    # --- POST-PROCESAMIENTO MINIMALISTA ---
    # Eliminados medianBlur y morphologyEx. Pasamos directo a limpieza.
    pred_final_eval = limpieza_minimalista(pred_class, raw_img.shape[0])

    # --- MÉTRICAS IOU ---
    ious_crudos = calculate_iou(true_class, pred_class, NUM_CLASSES)
    iou_disco_crudo.append(ious_crudos[0])
    iou_copa_cruda.append(ious_crudos[1])

    ious_finales = calculate_iou(true_class, pred_final_eval, NUM_CLASSES)
    iou_disco_final.append(ious_finales[0])
    iou_copa_final.append(ious_finales[1])

    # --- NUEVA MÉTRICA CLÍNICA (CDR) ---
    cdr_real = calcular_cdr_mejorado(true_class)

    # IMPORTANTE: Evaluamos la predicción cruda (pred_class) porque aún tiene todos los fragmentos
    cdr_pred = calcular_cdr_mejorado(pred_class)

    cdr_reales.append(cdr_real)
    cdr_predichos.append(cdr_pred)
    errores_cdr.append(abs(cdr_real - cdr_pred))

print("\n--- RESULTADOS IOU (PÍXELES) ---")
print(f"Crudo - Disco Óptico: {np.mean(iou_disco_crudo):.4f} | Copa Óptica: {np.mean(iou_copa_cruda):.4f}")
print(f"FINAL - Disco Óptico: {np.mean(iou_disco_final):.4f} | Copa Óptica: {np.mean(iou_copa_final):.4f}")

print("\n--- RESULTADOS CLÍNICOS (CDR) ---")
print(f"CDR Real Promedio:     {np.mean(cdr_reales):.4f}")
print(f"CDR Predicho Promedio: {np.mean(cdr_predichos):.4f}")
print(f"Margen de Error Promedio (MAE): {np.mean(errores_cdr):.4f} (Lo ideal es < 0.05)")

print("\nVisualización de ejemplo del Test Set:")
idx_random = random.randint(0, len(X_test_paths) - 1)
test_raw_img = preprocess_image_raw(X_test_paths[idx_random])
test_mask = preprocess_multiclass_mask(Y_test_paths[idx_random])

visualize_prediction(model, test_raw_img, test_mask)